In [2]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Cell 2 — Imports
import numpy as np
import torch
print('GPU:', torch.cuda.is_available())

GPU: True


In [4]:
# Cell 3 — Install transformers
!pip install -q transformers==4.38.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.


In [5]:
# Cell 4 — Load WavLM-Base model
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import torch

device = 'cuda'

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base')
model             = WavLMModel.from_pretrained('microsoft/wavlm-base').to(device)
model.eval()

print('Model ready ✅')
print('Hidden size: 768')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of the model checkpoint at microsoft/wavlm-base were not used when initializing WavLMModel: ['encoder.pos_conv_embed.conv.weight_g', 'encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing WavLMModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing WavLMModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of WavLMModel were not initialized from the model checkpoint at microsoft/wavlm-base and are newly initialized: ['encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inf

Model ready ✅
Hidden size: 768


In [6]:
# Cell 5 — Define paths
import os
import pandas as pd

DRIVE_ROOT           = '/content/drive/MyDrive/Hackenza_MUPlovers'
DATA_PATH            = os.path.join(DRIVE_ROOT, 'data')
PROCESSED_AUDIO_PATH = os.path.join(DATA_PATH, 'processed')
CHUNK_INDEX_PATH     = os.path.join(DATA_PATH, 'chunk_index.csv')

# Saves to embeddings_base/ — keeps other folders safe
EMBED_SAVE_PATH = os.path.join(DRIVE_ROOT, 'cache', 'embeddings_base')
os.makedirs(EMBED_SAVE_PATH, exist_ok=True)

chunk_df = pd.read_csv(CHUNK_INDEX_PATH)

print('Drive root          :', DRIVE_ROOT)
print('Processed audio path:', PROCESSED_AUDIO_PATH)
print('Chunk index path    :', CHUNK_INDEX_PATH)
print('Embedding save path :', EMBED_SAVE_PATH)
print('Total chunks        :', len(chunk_df))
print('Unique files        :', chunk_df['file_id'].nunique())

Drive root          : /content/drive/MyDrive/Hackenza_MUPlovers
Processed audio path: /content/drive/MyDrive/Hackenza_MUPlovers/data/processed
Chunk index path    : /content/drive/MyDrive/Hackenza_MUPlovers/data/chunk_index.csv
Embedding save path : /content/drive/MyDrive/Hackenza_MUPlovers/cache/embeddings_base
Total chunks        : 5179
Unique files        : 160


In [7]:
# Cell 6 — Define chunk embedding extraction
def extract_chunk_embedding(chunk_audio, feature_extractor, model, device='cuda'):
    """
    Extracts embedding from a single audio chunk.
    Uses last hidden state of WavLM-Base.
    Returns tensor of shape [768]
    """
    inputs = feature_extractor(
        chunk_audio.cpu().numpy(),
        sampling_rate=16000,
        return_tensors='pt'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.last_hidden_state  # [1, frames, 768]
    embedding     = hidden_states.mean(dim=1)  # [1, 768]
    embedding     = embedding.squeeze(0)       # [768]

    return embedding.cpu()

print('✅ Chunk embedding function defined!')

✅ Chunk embedding function defined!


In [8]:
# Cell 7 — Define file level embedding extraction
import torchaudio

def extract_embeddings_for_file(file_id):
    """
    Extracts embeddings for all chunks of one file.
    Returns tensor of shape [T, 768]
    """
    audio_path   = os.path.join(PROCESSED_AUDIO_PATH, f'{file_id}.wav')
    waveform, sr = torchaudio.load(audio_path)

    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.squeeze(0)

    file_chunks = chunk_df[chunk_df['file_id'] == int(file_id)]
    embeddings  = []

    for _, row in file_chunks.iterrows():
        start_sample = int(row['start_sample'])
        end_sample   = int(row['end_sample'])
        chunk_audio  = waveform[start_sample:end_sample]

        # Ensure exactly 3 seconds = 48000 samples
        expected_len = 48000
        if len(chunk_audio) < expected_len:
            padding     = expected_len - len(chunk_audio)
            chunk_audio = torch.nn.functional.pad(chunk_audio, (0, padding))

        embedding = extract_chunk_embedding(
            chunk_audio,
            feature_extractor,
            model,
            device=device
        )
        embeddings.append(embedding)

    embeddings = torch.stack(embeddings)  # [T, 768]

    assert embeddings.shape[0] == len(file_chunks), \
        f'Chunk count mismatch for {file_id}!'

    return embeddings

print('✅ File embedding function defined!')

✅ File embedding function defined!


In [9]:
# Cell 8 — Test on one file
test_file = '288'

print(f'Testing on file {test_file}...')
emb = extract_embeddings_for_file(test_file)

print(f'✅ Embedding shape: {emb.shape}')
print(f'   Expected       : [T, 768]')

assert emb.shape[1] == 768, '❌ Wrong dim! Should be 768'
print('\n✅ Test passed!')

Testing on file 288...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


✅ Embedding shape: torch.Size([24, 768])
   Expected       : [T, 768]

✅ Test passed!


In [10]:
# Cell 9 — Extract embeddings for ALL 160 files
from tqdm import tqdm

all_files = sorted([
    f.replace('.wav', '')
    for f in os.listdir(PROCESSED_AUDIO_PATH)
    if f.endswith('.wav')
])

print(f'Total files: {len(all_files)}')
print(f'Saving to  : {EMBED_SAVE_PATH}')
print(f'Each file will be [T, 768]\n')

failed = []

for file_id in tqdm(all_files):
    save_path = os.path.join(EMBED_SAVE_PATH, f'{file_id}.npy')

    if os.path.exists(save_path):
        continue

    try:
        emb = extract_embeddings_for_file(file_id)
        np.save(save_path, emb.cpu().numpy())
    except Exception as e:
        print(f'❌ Failed {file_id}: {e}')
        failed.append(file_id)

print(f'\n✅ Done! {len(all_files)-len(failed)}/{len(all_files)} files saved')
if failed:
    print('❌ Failed:', failed)

Total files: 160
Saving to  : /content/drive/MyDrive/Hackenza_MUPlovers/cache/embeddings_base
Each file will be [T, 768]



100%|██████████| 160/160 [02:38<00:00,  1.01it/s]


✅ Done! 160/160 files saved


In [12]:
# Cell 10 — Verify all files are [T, 768]
print('Verifying all files...')

all_saved = os.listdir(EMBED_SAVE_PATH)
issues    = []
shapes    = []

for fname in tqdm(all_saved):
    if not fname.endswith('.npy'):
        continue
    emb = np.load(os.path.join(EMBED_SAVE_PATH, fname))
    if emb.shape[1] != 768:
        issues.append(f'{fname} → {emb.shape}')
    else:
        shapes.append(emb.shape)

print(f'\n=== Verification ===')
print(f'Total files    : {len(shapes)}')
print(f'All dim = 768  : {len(issues) == 0}')

if shapes:
    T_vals = [s[0] for s in shapes]
    print(f'T range        : min={min(T_vals)}, max={max(T_vals)}')

if issues:
    print('❌ Issues:', issues)
else:
    print('\n✅ All files are [T, 768]!')
    print('→ Feature dim after assembly = 768 + 10 + 5 = 783')
    print('→ Tell Aalu to run Feature Assembly with EMB_DIR = cache/embeddings_base/')

Verifying all files...


100%|██████████| 160/160 [00:00<00:00, 318.41it/s]


=== Verification ===
Total files    : 160
All dim = 768  : True
T range        : min=12, max=267

✅ All files are [T, 768]!
→ Feature dim after assembly = 768 + 10 + 5 = 783
→ Tell Aalu to run Feature Assembly with EMB_DIR = cache/embeddings_base/
